# Notebook 1 - Prepare experiment data

The JSON manifests in `experiments/` are the source of truth for the experiment grid. This notebook materializes generated instances from `small_exact.json` and `artificial.json` under `data/experiment_instances/` for inspection and reuse.

The OR-Library inputs referenced by `benchmarks.json` are static benchmark files, not synthetic data. They are kept under `data/benchmarks/` and are checked below rather than regenerated.

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from data_generator import generate_instance, instance_to_dict

EXPERIMENTS_DIR = REPO_ROOT / "experiments"
DATA_DIR = REPO_ROOT / "data" / "experiment_instances"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_MANIFESTS = (EXPERIMENTS_DIR / "small_exact.json", EXPERIMENTS_DIR / "artificial.json")

In [ ]:
materialized = []
for manifest_path in GENERATED_MANIFESTS:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest_dir = DATA_DIR / manifest_path.stem
    manifest_dir.mkdir(parents=True, exist_ok=True)
    for specification in manifest["instances"]:
        generation = specification.get("generate")
        if generation is None:
            continue
        instance = generate_instance(**generation)
        payload = instance_to_dict(instance)
        payload["metadata"] = {
            "experiment": manifest_path.stem,
            "name": specification["name"],
            "family": specification.get("family", ""),
            **generation,
        }
        output_path = manifest_dir / f"{specification['name']}.json"
        output_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
        materialized.append({"experiment": manifest_path.stem, "name": specification["name"], "path": str(output_path.relative_to(REPO_ROOT))})

index_path = DATA_DIR / "index.json"
index_path.write_text(json.dumps(materialized, indent=2) + "\n", encoding="utf-8")
print(f"Materialized {len(materialized)} generated instances")
print(f"Index: {index_path.relative_to(REPO_ROOT)}")

In [ ]:
benchmark_manifest = json.loads(
    (EXPERIMENTS_DIR / "benchmarks.json").read_text(encoding="utf-8")
)
missing_benchmarks = [
    (item["name"], (EXPERIMENTS_DIR / item["path"]).resolve())
    for item in benchmark_manifest["instances"]
    if "path" in item and not (EXPERIMENTS_DIR / item["path"]).is_file()
]
if missing_benchmarks:
    raise FileNotFoundError(f"Missing benchmark inputs: {missing_benchmarks}")
print(f"Verified {len(benchmark_manifest['instances'])} static benchmark inputs in data/benchmarks/")

In [ ]:
from collections import Counter

for manifest_path in GENERATED_MANIFESTS:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    generated = [item for item in manifest["instances"] if "generate" in item]
    sizes = Counter(item["generate"]["num_elements"] for item in generated)
    densities = sorted({item["generate"]["density"] for item in generated})
    print(f"{manifest_path.stem}: {len(generated)} cases; sizes={dict(sizes)}; densities={densities}")

print(f"Total JSON files: {len(materialized)}")